# 03 Feature Engineering

In this notebook, we perform feature engineering after removing extreme price outliers.

Input dataset:
- dataset/price_outliers_removed_dataset.csv

Main goals:
1. Create useful features for analysis and modeling.
2. Analyze the relationship between price and demand.
3. Save a new dataset after feature engineering.

Important note:
The previous notebook handled price outliers using the 99th percentile method.
This notebook does not remove additional outliers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv("../dataset/price_outliers_removed_dataset.csv")

print("Dataset loaded successfully")
print("Shape:", df.shape)

df.head()

In [ ]:
df.info()
df.columns

## Initial Check

Before creating new features, we check missing values and duplicated rows to make sure the dataset is still clean after the previous preparation step.

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicated rows:", df.duplicated().sum())

## Feature Engineering

We will create new features that can help us better understand product price behavior and the relationship between price and demand.

The new features include:
- total_order_item_value
- freight_ratio
- product_volume_cm3
- price_range
- freight_level
- product_size_level

### total_order_item_value

This feature represents the total cost paid by the customer for one item, including both product price and freight cost.

It was created because customer purchase decisions are affected by the final cost, not only the product price. A product with a low price but high shipping cost may be less attractive to customers.

In [ ]:
df["total_order_item_value"] = df["price"] + df["freight_value"]

df[["price", "freight_value", "total_order_item_value"]].head()

### freight_ratio

This feature measures the shipping cost relative to the product price.

It was created because the shipping cost alone may not fully explain customer behavior. For example, a freight value of 50 may be acceptable for an expensive product, but too high for a cheap product.

This feature helps us understand whether the shipping cost is high or low compared to the product price, which may affect customer demand.

In [ ]:
df["freight_ratio"] = df["freight_value"] / df["price"]

df["freight_ratio"] = df["freight_ratio"].replace([np.inf, -np.inf], np.nan)
df["freight_ratio"] = df["freight_ratio"].fillna(0)
df["freight_ratio_percent"] = df["freight_ratio"].apply(lambda x: f"{x * 100:.2f}%")

df[["price", "freight_value", "freight_ratio","freight_ratio_percent"]].head()

### product_volume_cm3

This feature represents the total physical volume of the product, calculated using product length, height, and width.

It was created because product dimensions individually may be less informative than the overall product volume. Product volume can affect shipping cost, storage requirements, and possibly product pricing.

In [ ]:
df["product_volume_cm3"] = (
    df["product_length_cm"] * 
    df["product_height_cm"] * 
    df["product_width_cm"]
)

df[["product_length_cm", "product_height_cm", "product_width_cm", "product_volume_cm3"]].head()

### price_range

This feature converts the continuous price column into price categories such as Very Low, Low, Medium, High, and Very High.

It was created to make price analysis easier and more interpretable. Instead of analyzing each price value separately, we can compare demand across different price ranges.

This is especially useful for analyzing the relationship between price and demand.

In [ ]:
df["price_range"] = pd.cut(
    df["price"],
    bins=[0, 50, 100, 200, 500, df["price"].max()],
    labels=["Very Low", "Low", "Medium", "High", "Very High"],
    include_lowest=True
)
df[['price','price_range']].head()

### freight_level

This feature converts the freight value into shipping cost categories.

It was created to make shipping cost analysis easier. Freight cost can influence customer purchasing decisions, especially when the shipping cost is high compared to the product price.

This feature helps compare demand and pricing behavior across different freight levels.

In [ ]:
df["freight_level"] = pd.cut(
    df["freight_value"],
    bins=[0, 10, 25, 50, df["freight_value"].max()],
    labels=["Low Freight", "Medium Freight", "High Freight", "Very High Freight"],
    include_lowest=True
)

df[["freight_value", "freight_level"]].head()

### product_size_level

This feature groups products into size categories: Small, Medium, and Large, based on product volume.

It was created to make product size analysis easier. Instead of using raw volume values, we can compare pricing, shipping cost, and demand across different product size groups.

Quantiles were used because they divide products based on the actual distribution of the dataset.

In [ ]:
df["product_size_level"] = pd.cut(
    df["product_volume_cm3"],
    bins=[0, 
          df["product_volume_cm3"].quantile(0.33),
          df["product_volume_cm3"].quantile(0.66),
          df["product_volume_cm3"].max()],
    labels=["Small", "Medium", "Large"],
    include_lowest=True
)

df[["product_volume_cm3", "product_size_level"]].head()

## Demand Feature Creation

Demand is represented as the number of times a product appears in orders.

We create:
- product_demand_count: number of times each product appears in the dataset.
- category_demand_count: number of orders/items per product category.

In [ ]:
df["product_demand_count"] = df.groupby("product_id")["product_id"].transform("count")

df[['product_id','product_demand_count']].head()

Note: `product_demand_count` represents how many times each product appears in the dataset. Since the dataset is at the order-item level, products with higher demand appear in multiple rows.

In [ ]:
df["category_demand_count"] = (
    df.groupby("product_category_name_english")["product_category_name_english"]
      .transform("count")
)
df[["product_category_name_english", "price", "category_demand_count"]].head()


In [ ]:
df["category_avg_price"] = df.groupby("product_category_name_english")["price"].transform("mean")
df["category_median_price"] = df.groupby("product_category_name_english")["price"].transform("median")
df["price_vs_category_median"] = df["price"] - df["category_median_price"]

df[[
    "product_category_name_english",
    "price",
    "category_avg_price",
    "category_median_price",
    "price_vs_category_median"
]].round(2).head()

### Interpretation of price_vs_category_median

The `price_vs_category_median` feature shows how each product price compares to the typical price of its category.

- If the value is positive, the product is more expensive than the category median price.
- If the value is negative, the product is cheaper than the category median price.
- If the value is close to zero, the product price is close to the normal price range of its category.

This feature is useful because the same price may be considered cheap in one category and expensive in another category. Therefore, comparing a product price with its category median gives better context than looking at the raw price alone.

## Price and Demand Relationship

In this section, we analyze the relationship between product price and demand.

Demand is represented by:
- product_demand_count
- category_demand_count

We want to understand whether cheaper products tend to have higher demand, and whether expensive products have lower demand.

In [ ]:
price_demand_summary = df.groupby("price_range", observed=True).agg(
        avg_price=("price", "mean"),
        median_price=("price", "median"),
        avg_demand=("product_demand_count", "mean"),
        total_items=("product_demand_count", "count")
    ).reset_index()

price_demand_summary["avg_price"] = price_demand_summary["avg_price"].round(2)
price_demand_summary["median_price"] = price_demand_summary["median_price"].round(2)
price_demand_summary["avg_demand"] = price_demand_summary["avg_demand"].round(2)

price_demand_summary

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=price_demand_summary, x="price_range", y="avg_demand")
plt.title("Average Product Demand by Price Range")
plt.xlabel("Price Range")
plt.ylabel("Average Demand")
plt.xticks(rotation=45)
plt.show()

### Observation

The chart shows that the Low price range has the highest average product demand, while demand generally decreases as the price range becomes higher.

However, the Very Low range does not have the highest demand. This means that cheaper products are not always the most demanded products. Demand may also be affected by other factors such as product category, product popularity, shipping cost, and customer need.

Overall, the result suggests a weak negative relationship between price and demand, where higher-priced products tend to have lower demand, but price alone does not fully explain demand.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="price", y="product_demand_count", alpha=0.2)
plt.title("Relationship Between Price and Product Demand")
plt.xlabel("Price")
plt.ylabel("Product Demand Count")
plt.show()

This scatter plot shows the relationship between product price and product demand.
It helps identify whether products with lower prices are ordered more frequently than expensive products.

In [ ]:
price_demand_corr = df["price"].corr(df["product_demand_count"])
print("Correlation between price and product demand:", round(price_demand_corr, 4))

In [ ]:
category_demand_summary = df.groupby("product_category_name_english").agg(
    median_price=("price", "median"),
    avg_price=("price", "mean"),
    total_demand=("order_id", "count") if "order_id" in df.columns else ("price", "count")
    ).reset_index()

category_demand_summary = category_demand_summary.sort_values("total_demand", ascending=False)

category_demand_summary.head(10)


In [ ]:
top_categories = category_demand_summary.head(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_categories, x="total_demand", y="product_category_name_english")
plt.title("Top 10 Product Categories by Demand")
plt.xlabel("Total Demand")
plt.ylabel("Product Category")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=category_demand_summary,
    x="median_price",
    y="total_demand",
    alpha=0.7
    )
plt.title("Category Median Price vs Total Demand")
plt.xlabel("Median Price")
plt.ylabel("Total Demand")
plt.show()

This chart compares the median price of each category with its total demand.
It helps us understand whether categories with lower median prices receive more demand than expensive categories.

In [ ]:
print("Final dataset shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
missing_after_fe = df.isnull().sum()
missing_after_fe[missing_after_fe > 0]

In [ ]:
output_path = "../dataset/feature_engineered_dataset.csv"

df.to_csv(output_path, index=False)

print("Feature engineered dataset saved successfully at:")
print(output_path)
print("Final shape:", df.shape)

## Summary

In this notebook, we created several new features to support data analysis and future modeling.

Created features:
- total_order_item_value
- freight_ratio
- product_volume_cm3
- price_range
- freight_level
- product_size_level
- product_demand_count
- category_demand_count
- category_avg_price
- category_median_price
- price_vs_category_median

The correlation between price and product demand was -0.064, which indicates a very weak negative relationship. This means that higher prices may be associated with slightly lower demand, but price alone is not enough to explain product demand.

We also analyzed the relationship between price and demand using:
- Price range vs average demand
- Price vs product demand scatter plot
- Category median price vs total demand

Finally, the feature-engineered dataset was saved as:
dataset/feature_engineered_dataset.csv